In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
from src.azure_sql import (
    get_engine,
    test_connection,
    read_sql,
    write_dataframe
)

from src.transformations import (
    transform_accounts,
    transform_customers,
    transform_journal_entries
)

In [3]:
engine = get_engine()
test_connection(engine)

print("Azure SQL: OK")

Azure SQL: OK


In [4]:
latest_batch = read_sql(
    """
    SELECT TOP 1 batch_id
    FROM bronze.qbo_accounts_raw
    ORDER BY extracted_at DESC
    """,
    engine
)

batch_id = latest_batch.loc[0, "batch_id"]

print("Batch:", batch_id)

Batch: 09f210a6-6c87-4714-87e9-c8ffd1bf6a23


In [5]:
df_accounts_raw = read_sql(
    f"""
    SELECT *
    FROM bronze.qbo_accounts_raw
    WHERE batch_id = '{batch_id}'
    """,
    engine
)

df_customers_raw = read_sql(
    f"""
    SELECT *
    FROM bronze.qbo_customers_raw
    WHERE batch_id = '{batch_id}'
    """,
    engine
)

df_journals_raw = read_sql(
    f"""
    SELECT *
    FROM bronze.qbo_journal_entries_raw
    WHERE batch_id = '{batch_id}'
    """,
    engine
)

print("Accounts:", len(df_accounts_raw))
print("Customers:", len(df_customers_raw))
print("Journals:", len(df_journals_raw))

Accounts: 89
Customers: 411
Journals: 39


In [6]:
dim_account = transform_accounts(
    df_accounts_raw
)

dim_customer = transform_customers(
    df_customers_raw
)

fact_gl = transform_journal_entries(
    df_journals_raw,
    journal_min=202309,
    journal_max=202608
)

print("dim_account:", len(dim_account))
print("dim_customer:", len(dim_customer))
print("fact_gl:", len(fact_gl))

dim_account: 89
dim_customer: 411
fact_gl: 1116


In [7]:
display(dim_account.head())
display(dim_customer.head())
display(fact_gl.head())

,account_id,account_name,account_number,account_type,account_subtype,classification,fully_qualified_name,active
0,69,Accounting,None,Expense,LegalProfessionalFees,Expense,Legal & Professional Fees:Accounting,True
1,33,Accounts Payable (A/P),None,Accounts Payable,AccountsPayable,Liability,Accounts Payable (A/P),True
2,84,Accounts Receivable (A/R),None,Accounts Receivable,AccountsReceivable,Asset,Accounts Receivable (A/R),True
3,7,Advertising,None,Expense,AdvertisingPromotional,Expense,Advertising,True
4,89,Arizona Dept. of Revenue Payable,None,Other Current Liability,GlobalTaxPayable,Liability,Arizona Dept. of Revenue Payable,True


,customer_id,display_name,company_name,email,phone,city,state,postal_code,country,active
0,1,Amy's Bird Sanctuary,Amy's Bird Sanctuary,Birds@Intuit.com,(650) 555-3311,Bayshore,CA,94326,NaN,True
1,411,Apex Analytics 354,Apex Analytics 354,billing.c0354@apex-analytics-354.example,+1 212-555-1354,New York,NY,10001,United States,True
2,254,Apex Capital 197,Apex Capital 197,billing.c0197@apex-capital-197.example,+33 1 55 1197,Paris,Ile-de-France,75001,France,True
3,287,Apex Foods 230,Apex Foods 230,billing.c0230@apex-foods-230.example,+1 415-555-1230,San Francisco,CA,94105,United States,True
4,167,Apex Group 110,Apex Group 110,billing.c0110@apex-group-110.example,+1 604-555-1110,Vancouver,BC,V6B 1A1,Canada,True


,journal_id,journal_no,txn_date,line_id,description,account_id,account_name,posting_type,amount,signed_amount,year,month,year_month
0,180,202608,2026-08-01,0,[Revenue] Recurring subscription revenue,1,Services,Credit,1665150.46,-1665150.46,2026,8,2026-08
1,180,202608,2026-08-01,1,[Revenue] Implementation and onboarding fees,5,Fees Billed,Credit,47913.34,-47913.34,2026,8,2026-08
2,180,202608,2026-08-01,2,[Revenue] Customer credits and discounts,86,Discounts given,Debit,9094.02,9094.02,2026,8,2026-08
3,180,202608,2026-08-01,3,[Engineering] Cloud hosting and infrastructure,80,Cost of Goods Sold,Debit,241691.62,241691.62,2026,8,2026-08
4,180,202608,2026-08-01,4,[Customer Success] Customer support platforms ...,80,Cost of Goods Sold,Debit,57009.97,57009.97,2026,8,2026-08


In [8]:
print(
    "CloudFlow journals:",
    fact_gl["journal_no"].nunique()
)

CloudFlow journals: 36


In [9]:
write_dataframe(
    dim_account,
    table="dim_account",
    schema="silver",
    if_exists="replace",
    engine=engine
)

write_dataframe(
    dim_customer,
    table="dim_customer",
    schema="silver",
    if_exists="replace",
    engine=engine
)

write_dataframe(
    fact_gl,
    table="fact_gl",
    schema="silver",
    if_exists="replace",
    engine=engine
)

print("Silver load: OK")

Silver load: OK


In [10]:
check = read_sql(
    """
    SELECT 'dim_account' AS table_name, COUNT(*) AS rows
    FROM silver.dim_account

    UNION ALL

    SELECT 'dim_customer', COUNT(*)
    FROM silver.dim_customer

    UNION ALL

    SELECT 'fact_gl', COUNT(*)
    FROM silver.fact_gl
    """,
    engine
)

display(check)

,table_name,rows
0,dim_account,89
1,dim_customer,411
2,fact_gl,1116
